# MiniMind: learn training from random initialization to inference

This notebook calls the repository's native PyTorch trainer scripts through `colab/minimind_colab.py`. Run each cell in order. First complete the small `micro` path; only then enable the full mini-data `zero` path.

In [7]:
# Set these to the branch that contains this colab/ directory.
REPOSITORY_URL = 'https://github.com/wangzheng422/minimind.git'
REPOSITORY_REF = 'wzh-main'
ROOT = '/content/minimind'
RUN_ZERO_PROFILE = False
RUN_TOKENIZER_EXPERIMENT = False
USE_GOOGLE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/colab/minimind/'


In [8]:
from pathlib import Path
import subprocess

root = Path(ROOT)
if not root.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF, REPOSITORY_URL, ROOT], check=True)
elif not (root / '.git').exists():
    raise RuntimeError(f'{ROOT} exists but is not a Git checkout')
else:
    subprocess.run(['git', '-C', ROOT, 'fetch', '--depth', '1', 'origin', REPOSITORY_REF], check=True)
    subprocess.run(['git', '-C', ROOT, 'checkout', '--detach', 'FETCH_HEAD'], check=True)
%cd {ROOT}
!python colab/minimind_colab.py setup
COLAB_PYTHON = f'{ROOT}/.colab-venv/bin/python'


/content/minimind
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 124.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 129.3 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 

In [ ]:
!{COLAB_PYTHON} colab/minimind_colab.py preflight --require-a100


## 1. Tokenizer, model tensors, and next-token labels

The next cells expose the same tokenizer, causal logits, and shifted labels that the trainer uses. `loss: true` means the following token is a supervised target.

In [ ]:
!{COLAB_PYTHON} colab/minimind_colab.py lesson tokenizer --text '语言模型通过预测下一个 token 来学习文本。'
!{COLAB_PYTHON} colab/minimind_colab.py lesson model --profile micro


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !{COLAB_PYTHON} colab/minimind_colab.py restore --drive-dir {DRIVE_DIR} --allow-missing
    import subprocess, threading, time
    from pathlib import Path
    Path(ROOT, 'logs').mkdir(exist_ok=True)
    def backup_loop():
        while True:
            with open(f'{ROOT}/logs/drive-backup.log', 'a') as log:
                subprocess.run([COLAB_PYTHON, 'colab/minimind_colab.py', 'backup', '--drive-dir', DRIVE_DIR], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
            time.sleep(600)
    threading.Thread(target=backup_loop, daemon=True).start()
!{COLAB_PYTHON} colab/minimind_colab.py download --all
!{COLAB_PYTHON} colab/minimind_colab.py make-micro --rows 2048
if RUN_TOKENIZER_EXPERIMENT:
    !{COLAB_PYTHON} colab/minimind_colab.py tokenizer-experiment
!{COLAB_PYTHON} colab/minimind_colab.py lesson pretrain-labels --profile micro --rows 24
!{COLAB_PYTHON} colab/minimind_colab.py lesson sft-labels --profile micro --rows 48


## 2. Execute one real optimizer update

This is not a mock: it runs forward, cross-entropy, backward, gradient clipping, AdamW, and a second loss measurement on the same batch.

In [ ]:
!{COLAB_PYTHON} colab/minimind_colab.py one-step --profile micro --stage pretrain
!{COLAB_PYTHON} colab/minimind_colab.py train --profile micro --stage pretrain --resume
!{COLAB_PYTHON} colab/minimind_colab.py infer --profile micro --stage pretrain --prompt '人工智能是' --max-new-tokens 80
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} colab/minimind_colab.py backup --drive-dir {DRIVE_DIR}


In [ ]:
!{COLAB_PYTHON} colab/minimind_colab.py one-step --profile micro --stage sft
!{COLAB_PYTHON} colab/minimind_colab.py train --profile micro --stage sft --resume
!{COLAB_PYTHON} colab/minimind_colab.py infer --profile micro --stage sft --prompt '请解释什么是自注意力机制。' --max-new-tokens 128
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} colab/minimind_colab.py backup --drive-dir {DRIVE_DIR}


## 3. Optional: persist outputs in Google Drive

When enabled before data preparation, Drive is restored first, backed up every 10 minutes, and copied again after each stage. Training always remains on Colab's local disk.

In [ ]:
if USE_GOOGLE_DRIVE:
    !{COLAB_PYTHON} colab/minimind_colab.py backup --drive-dir {DRIVE_DIR}


## 4. Optional: MiniMind Zero reproduction

Set `RUN_ZERO_PROFILE = True` only after the micro route succeeds. This uses the complete official mini JSONL files and the repository's 768-hidden, 8-layer configuration.

In [ ]:
if RUN_ZERO_PROFILE:
    !{COLAB_PYTHON} colab/minimind_colab.py lesson model --profile zero
    !{COLAB_PYTHON} colab/minimind_colab.py train --profile zero --stage pretrain --resume
    !{COLAB_PYTHON} colab/minimind_colab.py infer --profile zero --stage pretrain --prompt '机器学习是一种' --max-new-tokens 128
    if USE_GOOGLE_DRIVE:
        !{COLAB_PYTHON} colab/minimind_colab.py backup --drive-dir {DRIVE_DIR}
    !{COLAB_PYTHON} colab/minimind_colab.py train --profile zero --stage sft --resume
    !{COLAB_PYTHON} colab/minimind_colab.py infer --profile zero --stage sft --prompt '请用通俗语言解释 Transformer。' --max-new-tokens 256
    if USE_GOOGLE_DRIVE:
        !{COLAB_PYTHON} colab/minimind_colab.py backup --drive-dir {DRIVE_DIR}
